# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# Bước 0 — Fold 0 và kiểm tra split
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_info = check_split(train_df, val_df, test_df, IMAGES_DIR)

print("Split sizes:", split_info["sizes"])
print("Overlap:", split_info["overlap"])
print("Union:", split_info["union"])
display(pd.DataFrame(split_info["per_class"]).T)

assert split_info["union"] == 17509
assert all(v == 0 for v in split_info["overlap"].values())


In [ ]:
# EDA — phân bố lớp
eda = pd.DataFrame({
    "train": train_df["Label"].value_counts().reindex(range(9), fill_value=0).to_numpy(),
    "val": val_df["Label"].value_counts().reindex(range(9), fill_value=0).to_numpy(),
    "test": test_df["Label"].value_counts().reindex(range(9), fill_value=0).to_numpy(),
}, index=CLASS_NAMES)

display(eda)

fig, ax = plt.subplots(figsize=(11, 5))
eda.plot(kind="bar", ax=ax)
ax.set_title("DeepWeeds fold 0 — class distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
eda_path = f"{WORK_DIR}/eda_class_distribution.png"
plt.savefig(eda_path, dpi=160, bbox_inches="tight")
plt.show()

weed_counts = eda["train"].iloc[:8]
print("Weed-class train imbalance max/min:", weed_counts.max() / weed_counts.min())
print("Negative / smallest weed class:", eda.loc["Negatives", "train"] / weed_counts.min())


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# Bước 1 — Backbone sweep
BACKBONES = {
    "B01": "resnet50",
    "B02": "resnext50_32x4d",
    "B03": "convnext_tiny",
    "B04": "deit_small_patch16_224",
    "B05": "efficientnet_b0",
}

# Đã chạy thực tế với:
# epochs=12, batch_size=64, seed=0, pretrained+finetune,
# img_size=224, basic augmentation, CE, AdamW.
RUN_BACKBONES = False  # False để mở notebook không train lại hàng giờ.

backbone_results = [
    dict(exp_id="B01", seed=0, best_epoch=10, macro_f1_val=0.8139385920, top1_val=0.8651813767, params_m=23.526473, gmac=4.109483008, train_sec=593.678707),
    dict(exp_id="B02", seed=0, best_epoch=11, macro_f1_val=0.8611587888, top1_val=0.8963153385, params_m=22.998345, gmac=4.257351680, train_sec=707.905963),
    dict(exp_id="B03", seed=0, best_epoch=11, macro_f1_val=0.9671756178, top1_val=0.9751499572, params_m=27.827049, gmac=4.469676288, train_sec=682.459966),
    dict(exp_id="B04", seed=0, best_epoch=12, macro_f1_val=0.9505161746, top1_val=0.9645815481, params_m=21.669129, gmac=4.250294016, train_sec=485.500617),
    dict(exp_id="B05", seed=0, best_epoch=10, macro_f1_val=0.8584108301, top1_val=0.8960297058, params_m=4.019077, gmac=0.398084384, train_sec=480.766133),
]
backbone_df = pd.DataFrame(backbone_results)
display(backbone_df)

# Nếu muốn chạy lại sweep trên Kaggle:
if RUN_BACKBONES:
    fresh = []
    for exp_id, backbone in BACKBONES.items():
        cfg = Config(
            exp_id=exp_id, seed=0, backbone=backbone,
            epochs=12, batch_size=64,
            images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
            out_dir=RUNS_DIR, pred_dir=PRED_DIR,
            init="finetune", aug="basic", loss="ce"
        )
        fresh.append(run(cfg))
    backbone_df = pd.DataFrame(fresh)

# Cấu hình được chọn để đi tiếp: ConvNeXt-Tiny (B03), dựa trên validation.
ABLATION_BACKBONE = "convnext_tiny"


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Bước 2 — Training ablation
ABLATIONS = [
    ("T00", dict(init="finetune", aug="basic", loss="ce")),
    ("T01", dict(init="scratch",  aug="basic", loss="ce")),
    ("T02", dict(init="frozen",   aug="basic", loss="ce")),
    ("T03", dict(init="finetune", aug="color", loss="ce")),
    ("T04", dict(init="finetune", aug="randaug", loss="ce")),
    ("T05", dict(init="finetune", aug="basic", loss="ls", label_smoothing=0.1)),
    ("T06", dict(init="finetune", aug="basic", loss="focal", focal_gamma=2.0)),
]

RUN_ABLATIONS = False  # Kết quả thực tế đã có; không train lại mặc định.

ablation_results = [
    dict(exp_id="T00", seed=0, best_epoch=11, macro_f1_val=0.9671756178, top1_val=0.9751499572, train_sec=632.935501),
    dict(exp_id="T01", seed=0, best_epoch=7,  macro_f1_val=0.313097,     top1_val=0.566410,      train_sec=645.113882),
    dict(exp_id="T02", seed=0, best_epoch=9,  macro_f1_val=0.855412,     top1_val=0.884890,      train_sec=442.230152),
    dict(exp_id="T03", seed=0, best_epoch=11, macro_f1_val=0.964376,     top1_val=0.971722,      train_sec=833.539241),
    dict(exp_id="T04", seed=0, best_epoch=12, macro_f1_val=0.965910,     top1_val=0.972865,      train_sec=639.901569),
    dict(exp_id="T05", seed=0, best_epoch=8,  macro_f1_val=0.966434,     top1_val=0.973436,      train_sec=640.274384),
    dict(exp_id="T06", seed=0, best_epoch=12, macro_f1_val=0.966771,     top1_val=0.974579,      train_sec=629.934318),
]
ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)

if RUN_ABLATIONS:
    fresh = []
    for exp_id, kw in ABLATIONS:
        cfg = Config(
            exp_id=exp_id, seed=0, backbone=ABLATION_BACKBONE,
            epochs=12, batch_size=64,
            images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
            out_dir=RUNS_DIR, pred_dir=PRED_DIR,
            **kw
        )
        fresh.append(run(cfg))
    ablation_df = pd.DataFrame(fresh)


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# Bước 3 — Inference, latency và calibration
# Kết quả thực tế đã chạy:
# - 1-view checkpoint benchmark trên ConvNeXt-Tiny, batch=1, 224px
# - FP32 và AMP latency
# - Temperature scaling fit trên VAL logits (không dùng TEST để fit T)
#
# Các phép TTA/multi-crop/ensemble khác được mô tả trong GUIDE nhưng
# không có kết quả thực nghiệm trong notebook đã cung cấp, nên không tự bịa số.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LATENCY_RESULTS = pd.DataFrame([
    dict(dtype="fp32", p50_ms=5.9353645, p95_ms=6.3220265, p99_ms=6.4253693, mean_ms=5.9503589, images_per_s=168.4816493),
    dict(dtype="amp",  p50_ms=8.1817460, p95_ms=8.7893702, p99_ms=9.0728090, mean_ms=8.2442927, images_per_s=122.2233005),
])
display(LATENCY_RESULTS)

# Temperature scaling:
TEMPERATURE = 2.0119092464
print("Temperature fitted on VAL:", TEMPERATURE)

# Code để benchmark lại khi có checkpoint T00/seed0:
# model = build_model("convnext_tiny", pretrained=True, num_classes=9, init="finetune").to(DEVICE)
# ckpt = torch.load(f"{RUNS_DIR}/T00/seed0/best.pt", map_location=DEVICE)
# model.load_state_dict(ckpt["model"]); model.eval()
# print(benchmark.latency_report(model, batch_size=1, img_size=224, dtype="fp32",
#                                device=str(DEVICE), warmup=10, iters=100))
# print(benchmark.latency_report(model, batch_size=1, img_size=224, dtype="amp",
#                                device=str(DEVICE), warmup=10, iters=100))


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# Bước 5 — Xuất results.xlsx
RESULTS_XLSX = f"{WORK_DIR}/results.xlsx"

summary_df = pd.DataFrame([
    ["Dataset", "Fold 0", "17,509 images"],
    ["Selected backbone", "B03", "convnext_tiny"],
    ["Final model", "F01", "finetune + basic aug + CE"],
    ["Seeds", "0,1,2", "3 seeds"],
    ["Temperature", "T", f"{TEMPERATURE:.6f}"],
    ["Latency", "FP32 p95", "6.322 ms"],
    ["Latency", "AMP p95", "8.789 ms"],
    ["F01 test", "macro-F1", "0.9680 ± 0.0017"],
    ["F01 test", "top-1", "0.9751 ± 0.0012"],
], columns=["Section", "Item", "Value"])

with pd.ExcelWriter(RESULTS_XLSX, engine="openpyxl") as writer:
    backbone_df.to_excel(writer, sheet_name="Backbones", index=False)
    ablation_df.to_excel(writer, sheet_name="Training", index=False)
    LATENCY_RESULTS.to_excel(writer, sheet_name="Latency", index=False)
    final_df.to_excel(writer, sheet_name="Final", index=False)
    per_class_f01.to_excel(writer, sheet_name="PerClass", index=False)
    final_test_summary.to_excel(writer, sheet_name="TestMetrics", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print("Saved:", RESULTS_XLSX)


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# Bước 5 — Xuất results.xlsx
RESULTS_XLSX = f"{WORK_DIR}/results.xlsx"

summary_df = pd.DataFrame([
    ["Dataset", "Fold 0", "17,509 images"],
    ["Selected backbone", "B03", "convnext_tiny"],
    ["Final model", "F01", "finetune + basic aug + CE"],
    ["Seeds", "0,1,2", "3 seeds"],
    ["Temperature", "T", f"{TEMPERATURE:.6f}"],
    ["Latency", "FP32 p95", "6.322 ms"],
    ["Latency", "AMP p95", "8.789 ms"],
    ["F01 test", "macro-F1", "0.9680 ± 0.0017"],
    ["F01 test", "top-1", "0.9751 ± 0.0012"],
], columns=["Section", "Item", "Value"])

with pd.ExcelWriter(RESULTS_XLSX, engine="openpyxl") as writer:
    backbone_df.to_excel(writer, sheet_name="Backbones", index=False)
    ablation_df.to_excel(writer, sheet_name="Training", index=False)
    LATENCY_RESULTS.to_excel(writer, sheet_name="Latency", index=False)
    final_df.to_excel(writer, sheet_name="Final", index=False)
    per_class_f01.to_excel(writer, sheet_name="PerClass", index=False)
    final_test_summary.to_excel(writer, sheet_name="TestMetrics", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print("Saved:", RESULTS_XLSX)
